# IFEST DAC 2026 — V8A Frozen Asymmetric Semantic Rescue

This notebook implements the agreed **V8A** only.

- Base system: exact frozen **V5**.
- Intervention direction: **0 → 1 only**.
- Eligible rows: `V5 == 0` and `V4 rule == v2_fallback`.
- Rescue score: frozen **V3 blend probability** only.
- Frozen threshold: **0.5817181386596663** (the original V3 threshold; no threshold search in V8A).
- Every V2/V4 structural decision and every existing V5 positive decision remains immutable.
- Validation reuses the exact prospective **DESIGN / LOCK** partition created before V7A. DESIGN is evaluated first; LOCK is opened only after the fixed V8A rule passes DESIGN.
- No model download and no training are performed. T4×2 is fully compatible but intentionally unnecessary for this post-processing version.

The frozen arrays below contain only prior-model predictions/scores and the previously frozen split metadata. They contain no hidden test labels.

In [1]:
import os, io, json, base64, hashlib, zipfile, warnings, shutil
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, average_precision_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 180)

# ---------------- V8A FROZEN CONFIG ----------------
SEED = 2026
LOCK_SEED = 260907
DESIGN_FOLDS = 3
BOOTSTRAP_REPS = 1000
BOOTSTRAP_SEED = LOCK_SEED + 77

V8_BLEND_MIN = 0.5817181386596663
REQUIRED_V4_RULE = 'v2_fallback'
DIRECTION = '0_to_1_only'

# DESIGN promotion gates agreed before implementation.
MIN_DESIGN_PRECISION = 0.80
MIN_POSITIVE_DESIGN_FOLDS = 3
MIN_DESIGN_GAIN = 0.0
MIN_DESIGN_BOOTSTRAP_LOWER = 0.0

# One-shot LOCK confirmation gates agreed before implementation.
MIN_LOCK_PRECISION = 0.70
MIN_LOCK_GAIN = 0.0
MIN_LOCK_BOOTSTRAP_POSITIVE = 0.80

EXPECTED_TRAIN_ROWS = 14397
EXPECTED_TEST_ROWS = 3603
EXPECTED_TRAIN_ID_SHA256 = '9abdabdd6cda9baac679b5cf004572e02326859fc2b13caa37af32c335e6c94a'
EXPECTED_TEST_ID_SHA256 = 'e4763272ee9a832ad2141f979a48701222a6651550ba2a88cf6c89357094064c'
EXPECTED_V5_SUBMISSION_SHA256 = '698790750562fda1dcc7eea78ac428d079312254b535f9aedf85fb240422dfdc'
EXPECTED_V5_OOF_CM = np.array([[1242,195],[381,12579]])
EXPECTED_V5_OOF_MACRO = 0.8946909330297752
EXPECTED_DESIGN_ROWS = 11548
EXPECTED_LOCK_ROWS = 2849
EXPECTED_DESIGN_CANDIDATES = 550
EXPECTED_DESIGN_RESCUES = 92
EXPECTED_TEST_CANDIDATES = 110
EXPECTED_TEST_RESCUES = 18
EXPECTED_V5_TEST_NEGATIVES = 379
EXPECTED_V8A_TEST_NEGATIVES = 361

RULE_NAMES = np.array(['v2_fallback','v2_graph_locked','test_content_canonical','test_title_canonical'], dtype=object)
OUTPUT_DIR = Path('/kaggle/working/ifest_v8a_output' if Path('/kaggle/working').exists() else '/mnt/data/ifest_v8a_output')
if OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('IFEST DAC 2026 — V8A Frozen Asymmetric Semantic Rescue')
print('Training: disabled by design')
print('Threshold:', repr(V8_BLEND_MIN))
try:
    import torch
    print(f'PyTorch={torch.__version__} | CUDA={torch.cuda.is_available()} | GPUs={torch.cuda.device_count()}')
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()): print(i, torch.cuda.get_device_name(i))
except Exception as e:
    print('Torch diagnostic unavailable:', repr(e))


IFEST DAC 2026 — V8A Frozen Asymmetric Semantic Rescue
Training: disabled by design
Threshold: 0.5817181386596663
PyTorch=2.10.0+cu128 | CUDA=True | GPUs=2
0 Tesla T4
1 Tesla T4


In [2]:
# Frozen prior outputs: V5 OOF/base state, V3 blend scores, exact V7A DESIGN/LOCK split, and V5 test state.
_FROZEN_B64 = '''...'''
_FROZEN_SHA256 = '86a14bf3ecb631369a8ad0617a451fa86c3e926b96126a80616cb7e48a151eb7'
_blob = base64.b64decode(_FROZEN_B64.encode())
assert hashlib.sha256(_blob).hexdigest() == _FROZEN_SHA256
_npz = np.load(io.BytesIO(_blob), allow_pickle=False)
train_v5_label = _npz['train_v5_label'].astype(np.int8)
train_v4_rule_code = _npz['train_v4_rule'].astype(np.int8)
train_v5_rescue = _npz['train_v5_rescue'].astype(bool)
train_v3_blend = _npz['train_v3_blend'].astype(np.float64)
train_partition = _npz['train_partition'].astype(np.int8)  # 0=DESIGN, 1=LOCK
train_design_fold = _npz['train_design_fold'].astype(np.int8)
train_component = _npz['train_component'].astype(np.int32)
test_v5_label = _npz['test_v5_label'].astype(np.int8)
test_v4_rule_code = _npz['test_v4_rule'].astype(np.int8)
test_v5_rescue = _npz['test_v5_rescue'].astype(bool)
test_v3_blend = _npz['test_v3_blend'].astype(np.float64)
del _npz, _blob

assert len(train_v5_label) == EXPECTED_TRAIN_ROWS
assert len(test_v5_label) == EXPECTED_TEST_ROWS
assert np.isfinite(train_v3_blend).all() and np.isfinite(test_v3_blend).all()
assert ((train_v3_blend >= 0) & (train_v3_blend <= 1)).all()
assert ((test_v3_blend >= 0) & (test_v3_blend <= 1)).all()
train_v4_rule = RULE_NAMES[np.clip(train_v4_rule_code, 0, 3)]
test_v4_rule = RULE_NAMES[np.clip(test_v4_rule_code, 0, 3)]
print('Frozen artifact loaded:', _FROZEN_SHA256)


Frozen artifact loaded: 86a14bf3ecb631369a8ad0617a451fa86c3e926b96126a80616cb7e48a151eb7


In [3]:
def find_competition_csv(filename, expected_cols, expected_rows):
    roots = [Path('/kaggle/input'), Path('/mnt/data/ifest_dac'), Path('/mnt/data/ifest_audit/penyisihan-ifest-2026-dac')]
    candidates = []
    for root in roots:
        if root.exists(): candidates.extend(root.rglob(filename))
    good = []
    for p in candidates:
        try:
            d = pd.read_csv(p, nrows=3)
            if d.columns.tolist() == expected_cols:
                rows = len(pd.read_csv(p, usecols=[expected_cols[0]]))
                if rows == expected_rows: good.append(p)
        except Exception:
            pass
    if not good:
        raise FileNotFoundError(f'Could not find a valid {filename} under Kaggle Input')
    good = sorted(good, key=lambda p: ('penyisihan-ifest-2026-dac' not in str(p).lower(), len(str(p))))
    return good[0]

def ids_sha256(series):
    return hashlib.sha256('\n'.join(series.astype(str).tolist()).encode()).hexdigest()

train_path = find_competition_csv('train.csv', ['id','title','content','label'], EXPECTED_TRAIN_ROWS)
test_path = find_competition_csv('test.csv', ['id','title','content'], EXPECTED_TEST_ROWS)
sample_path = find_competition_csv('sample_submission.csv', ['id','label'], EXPECTED_TEST_ROWS)
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)

assert train.columns.tolist() == ['id','title','content','label']
assert test.columns.tolist() == ['id','title','content']
assert sample.columns.tolist() == ['id','label']
assert sample.id.astype(str).tolist() == test.id.astype(str).tolist()
assert set(train.label.unique()) == {0,1}
assert ids_sha256(train.id) == EXPECTED_TRAIN_ID_SHA256, 'Train ID/order drift; frozen OOF arrays are unsafe to use.'
assert ids_sha256(test.id) == EXPECTED_TEST_ID_SHA256, 'Test ID/order drift; frozen test arrays are unsafe to use.'

# Frozen V5 parity checks. These are historical-base checks, not V8A model selection.
y = train.label.to_numpy(np.int8)
cm = confusion_matrix(y, train_v5_label, labels=[0,1])
base_macro = f1_score(y, train_v5_label, average='macro', labels=[0,1], zero_division=0)
assert np.array_equal(cm, EXPECTED_V5_OOF_CM), f'Frozen V5 OOF confusion parity failed: {cm.tolist()}'
assert abs(base_macro - EXPECTED_V5_OOF_MACRO) < 1e-12
assert int((test_v5_label == 0).sum()) == EXPECTED_V5_TEST_NEGATIVES
v5_submission = sample[['id']].copy(); v5_submission['label'] = test_v5_label.astype(int)
v5_sha = hashlib.sha256(v5_submission.to_csv(index=False).encode()).hexdigest()
assert v5_sha == EXPECTED_V5_SUBMISSION_SHA256, 'Frozen V5 test prediction parity failed.'

print('Dataset alignment and frozen V5 parity: OK')
print('train/test:', train.shape, test.shape)
print('Frozen V5 OOF Macro F1:', f'{base_macro:.9f}')
print('Frozen V5 OOF confusion:', cm.tolist())
print('Frozen V5 test negatives:', int((test_v5_label==0).sum()))


Dataset alignment and frozen V5 parity: OK
train/test: (14397, 4) (3603, 3)
Frozen V5 OOF Macro F1: 0.894690933
Frozen V5 OOF confusion: [[1242, 195], [381, 12579]]
Frozen V5 test negatives: 379


## Fixed V8A rescue policy

No parameter is selected in this notebook. The original V3 threshold is applied once, only to V5-negative pure-fallback rows. Structural rows are immutable.

In [4]:
def check_v8a_invariants(before, after, rules, rescue_mask):
    before = np.asarray(before, dtype=np.int8)
    after = np.asarray(after, dtype=np.int8)
    rules = np.asarray(rules, dtype=object)
    rescue_mask = np.asarray(rescue_mask, dtype=bool)
    if not (len(before) == len(after) == len(rules) == len(rescue_mask)):
        raise ValueError('V8A input length mismatch')
    if np.any((before == 1) & (after == 0)):
        raise RuntimeError('V8A attempted a forbidden 1->0 override')
    if np.any(rescue_mask & (rules != REQUIRED_V4_RULE)):
        raise RuntimeError('V8A attempted to override a structural V4 decision')
    if np.any(rescue_mask & (before != 0)) or np.any(rescue_mask & (after != 1)):
        raise RuntimeError('V8A rescue-direction invariant failed')
    if np.any((~rescue_mask) & (before != after)):
        raise RuntimeError('V8A changed a non-rescued row')
    return True

def apply_v8a_rescue(v5_labels, v4_rule, v3_blend):
    labels = np.asarray(v5_labels, dtype=np.int8).copy()
    rules = np.asarray(v4_rule, dtype=object)
    blend = np.asarray(v3_blend, dtype=float)
    if not (len(labels) == len(rules) == len(blend)):
        raise ValueError('V8A input length mismatch')
    if not np.isfinite(blend).all():
        raise ValueError('V8A received non-finite V3 scores')
    before = labels.copy()
    rescue = (before == 0) & (rules == REQUIRED_V4_RULE) & (blend >= V8_BLEND_MIN)
    labels[rescue] = 1
    check_v8a_invariants(before, labels, rules, rescue)
    return labels, rescue

def macro_f1(y_true, y_pred):
    return float(f1_score(y_true, y_pred, average='macro', labels=[0,1], zero_division=0))

def _counts4(y_true, y_pred):
    y_true=np.asarray(y_true,np.int8); y_pred=np.asarray(y_pred,np.int8)
    return np.array([
        np.sum((y_true==0)&(y_pred==0)),
        np.sum((y_true==0)&(y_pred==1)),
        np.sum((y_true==1)&(y_pred==0)),
        np.sum((y_true==1)&(y_pred==1)),
    ], dtype=np.int64)

def _macro_from_counts(c):
    c=np.asarray(c,dtype=float); tn,fp,fn,tp=np.moveaxis(c,-1,0)
    d0=2*tn+fp+fn; d1=2*tp+fp+fn
    f0=np.divide(2*tn,d0,out=np.zeros_like(tn,dtype=float),where=d0>0)
    f1=np.divide(2*tp,d1,out=np.zeros_like(tp,dtype=float),where=d1>0)
    return 0.5*(f0+f1)

def component_bootstrap_delta(y_true, base_pred, new_pred, components, reps=BOOTSTRAP_REPS, seed=BOOTSTRAP_SEED):
    y_true=np.asarray(y_true,np.int8); base_pred=np.asarray(base_pred,np.int8); new_pred=np.asarray(new_pred,np.int8); components=np.asarray(components)
    unique=np.unique(components)
    base_counts=np.vstack([_counts4(y_true[components==g], base_pred[components==g]) for g in unique])
    new_counts=np.vstack([_counts4(y_true[components==g], new_pred[components==g]) for g in unique])
    rng=np.random.default_rng(seed); deltas=[]; batch=100
    for start in range(0,reps,batch):
        n=min(batch,reps-start)
        draw=rng.integers(0,len(unique),size=(n,len(unique)))
        bc=base_counts[draw].sum(axis=1); nc=new_counts[draw].sum(axis=1)
        deltas.extend((_macro_from_counts(nc)-_macro_from_counts(bc)).tolist())
    return np.asarray(deltas,dtype=float)


## Prospective DESIGN evaluation

The exact V7A-era split is reused. Only DESIGN labels are used in this section. There is no threshold/coverage search.

In [5]:
design_mask = train_partition == 0
lock_mask = train_partition == 1
assert int(design_mask.sum()) == EXPECTED_DESIGN_ROWS
assert int(lock_mask.sum()) == EXPECTED_LOCK_ROWS
assert np.all(train_design_fold[lock_mask] == -1)
assert set(np.unique(train_design_fold[design_mask])) == {0,1,2}
assert set(train_component[design_mask]).isdisjoint(set(train_component[lock_mask])), 'Frozen DESIGN/LOCK component overlap detected.'
assert [int(((train_design_fold==f)&design_mask).sum()) for f in range(DESIGN_FOLDS)] == [3863,3869,3816]
_fold_by_component = pd.DataFrame({'component':train_component[design_mask],'fold':train_design_fold[design_mask]}).groupby('component').fold.nunique()
assert int(_fold_by_component.max()) == 1, 'A DESIGN component spans multiple validation folds.'

# Apply the already-frozen policy to DESIGN only.
design_base = train_v5_label[design_mask].copy()
design_rules = train_v4_rule[design_mask]
design_scores = train_v3_blend[design_mask]
design_y = y[design_mask]
design_new, design_rescue = apply_v8a_rescue(design_base, design_rules, design_scores)
design_candidate = (design_base == 0) & (design_rules == REQUIRED_V4_RULE)

n_cand = int(design_candidate.sum()); n_rescue = int(design_rescue.sum())
assert n_cand == EXPECTED_DESIGN_CANDIDATES, f'DESIGN candidate drift: {n_cand}'
assert n_rescue == EXPECTED_DESIGN_RESCUES, f'DESIGN rescue drift: {n_rescue}'

base_f1 = macro_f1(design_y, design_base)
new_f1 = macro_f1(design_y, design_new)
gain = new_f1 - base_f1
precision = float(design_y[design_rescue].mean()) if n_rescue else np.nan
cand_target = design_y[design_candidate]
cand_score = design_scores[design_candidate]
auc = float(roc_auc_score(cand_target, cand_score)) if np.unique(cand_target).size == 2 else np.nan
ap = float(average_precision_score(cand_target, cand_score)) if np.unique(cand_target).size == 2 else np.nan

fold_rows=[]
design_fold_local=train_design_fold[design_mask]
for f in range(DESIGN_FOLDS):
    m=design_fold_local==f
    b=design_base[m]; n=design_new[m]; yy=design_y[m]; rr=design_rescue[m]; cc=design_candidate[m]
    fold_rows.append({
        'fold':f,'rows':int(m.sum()),'candidate_rows':int(cc.sum()),'rescues':int(rr.sum()),
        'base_macro_f1':macro_f1(yy,b),'v8a_macro_f1':macro_f1(yy,n),
        'gain':macro_f1(yy,n)-macro_f1(yy,b),
        'rescue_precision':float(yy[rr].mean()) if rr.any() else np.nan,
    })
design_fold_metrics=pd.DataFrame(fold_rows)

boot_design = component_bootstrap_delta(
    design_y, design_base, design_new, train_component[design_mask], reps=BOOTSTRAP_REPS, seed=BOOTSTRAP_SEED
)
ci_low, ci_high = np.quantile(boot_design, [0.025,0.975])
boot_positive = float((boot_design > 0).mean())

DESIGN_OK = bool(
    gain > MIN_DESIGN_GAIN
    and precision >= MIN_DESIGN_PRECISION
    and int((design_fold_metrics.gain > 0).sum()) >= MIN_POSITIVE_DESIGN_FOLDS
    and ci_low > MIN_DESIGN_BOOTSTRAP_LOWER
)

design_metrics=pd.DataFrame([{
    'rows':int(design_mask.sum()),'candidate_rows':n_cand,'rescues':n_rescue,
    'candidate_positive_prevalence':float(cand_target.mean()),'candidate_roc_auc':auc,'candidate_average_precision':ap,
    'base_macro_f1':base_f1,'v8a_macro_f1':new_f1,'gain':gain,'rescue_precision':precision,
    'bootstrap_median_gain':float(np.median(boot_design)),'bootstrap_ci_low_95':float(ci_low),'bootstrap_ci_high_95':float(ci_high),
    'bootstrap_positive_fraction':boot_positive,'design_ok':DESIGN_OK,
}])

design_bootstrap=pd.DataFrame({'replicate':np.arange(BOOTSTRAP_REPS),'delta_macro_f1':boot_design})
print('DESIGN evaluation')
display(design_metrics)
display(design_fold_metrics)
print('DESIGN_OK:', DESIGN_OK)


DESIGN evaluation


,rows,candidate_rows,rescues,candidate_positive_prevalence,candidate_roc_auc,candidate_average_precision,base_macro_f1,v8a_macro_f1,gain,rescue_precision,bootstrap_median_gain,bootstrap_ci_low_95,bootstrap_ci_high_95,bootstrap_positive_fraction,design_ok
0,11548,550,92,0.485455,0.860814,0.845574,0.893167,0.907263,0.014096,0.913043,0.01414,0.00983,0.018905,1.0,True


,fold,rows,candidate_rows,rescues,base_macro_f1,v8a_macro_f1,gain,rescue_precision
0,0,3863,188,34,0.887812,0.905019,0.017207,0.970588
1,1,3869,176,25,0.898453,0.910584,0.012131,0.920000
2,2,3816,186,33,0.893641,0.906294,0.012652,0.848485


DESIGN_OK: True


## One-shot LOCK confirmation

This cell evaluates the already-frozen V8A rule on LOCK only if DESIGN passed. No parameter can change after this point.

In [6]:
LOCK_OPENED = False
LOCK_OK = False
lock_metrics = pd.DataFrame()
lock_bootstrap = pd.DataFrame()

if DESIGN_OK:
    LOCK_OPENED = True
    lock_base = train_v5_label[lock_mask].copy()
    lock_rules = train_v4_rule[lock_mask]
    lock_scores = train_v3_blend[lock_mask]
    lock_y = y[lock_mask]  # first V8A-specific access to LOCK labels
    lock_new, lock_rescue = apply_v8a_rescue(lock_base, lock_rules, lock_scores)
    lock_candidate = (lock_base == 0) & (lock_rules == REQUIRED_V4_RULE)

    lock_base_f1 = macro_f1(lock_y, lock_base)
    lock_new_f1 = macro_f1(lock_y, lock_new)
    lock_gain = lock_new_f1 - lock_base_f1
    lock_precision = float(lock_y[lock_rescue].mean()) if lock_rescue.any() else 0.0
    boot_lock = component_bootstrap_delta(
        lock_y, lock_base, lock_new, train_component[lock_mask], reps=BOOTSTRAP_REPS, seed=BOOTSTRAP_SEED+1
    )
    lock_boot_positive = float((boot_lock > 0).mean())
    lock_ci_low, lock_ci_high = np.quantile(boot_lock,[0.025,0.975])

    LOCK_OK = bool(
        lock_gain > MIN_LOCK_GAIN
        and lock_precision >= MIN_LOCK_PRECISION
        and lock_boot_positive >= MIN_LOCK_BOOTSTRAP_POSITIVE
    )
    lock_metrics = pd.DataFrame([{
        'rows':int(lock_mask.sum()),'candidate_rows':int(lock_candidate.sum()),'rescues':int(lock_rescue.sum()),
        'base_macro_f1':lock_base_f1,'v8a_macro_f1':lock_new_f1,'gain':lock_gain,
        'rescue_precision':lock_precision,'bootstrap_median_gain':float(np.median(boot_lock)),
        'bootstrap_ci_low_95':float(lock_ci_low),'bootstrap_ci_high_95':float(lock_ci_high),
        'bootstrap_positive_fraction':lock_boot_positive,'lock_ok':LOCK_OK,
    }])
    lock_bootstrap = pd.DataFrame({'replicate':np.arange(BOOTSTRAP_REPS),'delta_macro_f1':boot_lock})
    print('LOCK opened exactly once.')
    display(lock_metrics)
else:
    print('DESIGN gate failed. LOCK remains sealed; V8A is rejected.')

V8A_VALIDATED = bool(DESIGN_OK and LOCK_OPENED and LOCK_OK)
print('V8A prospective validation accepted:', V8A_VALIDATED)


LOCK opened exactly once.


,rows,candidate_rows,rescues,base_macro_f1,v8a_macro_f1,gain,rescue_precision,bootstrap_median_gain,bootstrap_ci_low_95,bootstrap_ci_high_95,bootstrap_positive_fraction,lock_ok
0,2849,122,16,0.901197,0.906627,0.00543,0.75,0.005286,-0.001951,0.012827,0.921,True


V8A prospective validation accepted: True


## Frozen test inference and export

Test inference contains no fitting or threshold selection. The notebook fail-closes to a clearly named V5 fallback artifact if prospective validation rejects V8A.

In [7]:
# Proposed V8A test intervention is deterministic and label-free.
test_candidate = (test_v5_label == 0) & (test_v4_rule == REQUIRED_V4_RULE)
proposed_v8a, test_rescue = apply_v8a_rescue(test_v5_label, test_v4_rule, test_v3_blend)

assert int(test_candidate.sum()) == EXPECTED_TEST_CANDIDATES, f'Test candidate drift: {int(test_candidate.sum())}'
assert int(test_rescue.sum()) == EXPECTED_TEST_RESCUES, f'Test rescue drift: {int(test_rescue.sum())}'
assert int((proposed_v8a == 0).sum()) == EXPECTED_V8A_TEST_NEGATIVES
assert not np.any(test_rescue & test_v5_rescue), 'V8A unexpectedly touched an existing V5 rescue row.'
assert np.all(test_v4_rule[test_rescue] == REQUIRED_V4_RULE)
assert np.all(test_v5_label[test_rescue] == 0) and np.all(proposed_v8a[test_rescue] == 1)

PROMOTED = V8A_VALIDATED
final_label = proposed_v8a.copy() if PROMOTED else test_v5_label.copy()

selection_rate_design = float(EXPECTED_DESIGN_RESCUES / EXPECTED_DESIGN_CANDIDATES)
selection_rate_test = float(test_rescue.sum() / test_candidate.sum())
print(f'Candidate selection rate DESIGN={selection_rate_design:.4%} | test={selection_rate_test:.4%}')
print('Test candidates:', int(test_candidate.sum()), '| proposed rescues:', int(test_rescue.sum()))
print('Promoted:', PROMOTED)

all_train_proposed, all_train_rescue = apply_v8a_rescue(train_v5_label, train_v4_rule, train_v3_blend)
train_scores = pd.DataFrame({
    'id':train.id,'label':y,'partition':np.where(lock_mask,'LOCK','DESIGN'),'design_fold':train_design_fold,
    'component':train_component,'v5_label':train_v5_label,'v4_rule':train_v4_rule,
    'v5_rescue':train_v5_rescue.astype(int),'v3_blend_prob':train_v3_blend,
    'v8a_candidate':((train_v5_label==0)&(train_v4_rule==REQUIRED_V4_RULE)).astype(int),
    'v8a_rescue':all_train_rescue.astype(int),'v8a_proposed_label':all_train_proposed.astype(int),
})

test_scores = pd.DataFrame({
    'id':test.id,'v5_label':test_v5_label,'v4_rule':test_v4_rule,'v5_rescue':test_v5_rescue.astype(int),
    'v3_blend_prob':test_v3_blend,'v8a_candidate':test_candidate.astype(int),'v8a_rescue':test_rescue.astype(int),
    'proposed_v8a_label':proposed_v8a.astype(int),'label':final_label.astype(int),
})
changed = test_scores.loc[test_scores.v8a_rescue==1].merge(test[['id','title','content']],on='id',how='left')

# Save validation artifacts before the submission decision.
artifacts = {
    'design_split_v8a.csv': pd.DataFrame({'id':train.id,'component':train_component,'partition':np.where(lock_mask,'LOCK','DESIGN'),'design_fold':train_design_fold}),
    'design_metrics_v8a.csv': design_metrics,
    'design_fold_metrics_v8a.csv': design_fold_metrics,
    'design_bootstrap_v8a.csv': design_bootstrap,
    'lock_metrics_v8a.csv': lock_metrics,
    'lock_bootstrap_v8a.csv': lock_bootstrap,
    'train_scores_v8a.csv': train_scores,
    'test_scores_v8a.csv': test_scores,
    'changed_rows_v8a.csv': changed,
}
for name,df in artifacts.items(): df.to_csv(OUTPUT_DIR/name,index=False)

if PROMOTED:
    submission = sample[['id']].copy(); submission['label'] = final_label.astype(int)
    assert submission.id.astype(str).tolist() == test.id.astype(str).tolist()
    assert set(submission.label.unique()).issubset({0,1}) and len(submission)==EXPECTED_TEST_ROWS
    submission.to_csv(OUTPUT_DIR/'submission_v8a.csv',index=False)
    submission.to_csv(OUTPUT_DIR/'submission.csv',index=False)
    print('SUBMIT artifact created:', OUTPUT_DIR/'submission_v8a.csv')
else:
    fallback = sample[['id']].copy(); fallback['label'] = test_v5_label.astype(int)
    fallback.to_csv(OUTPUT_DIR/'submission_v5_fallback_DO_NOT_SUBMIT_AS_V8A.csv',index=False)
    print('V8A rejected by prospective gate. No submission_v8a.csv was created.')

config = {
    'version':'v8a-frozen-asymmetric-semantic-rescue','training':False,'seed':SEED,'lock_seed':LOCK_SEED,
    'base':'exact-frozen-v5','direction':DIRECTION,'required_v4_rule':REQUIRED_V4_RULE,
    'score':'frozen-v3-blend','v3_blend_min':V8_BLEND_MIN,
    'design_gates':{'min_precision':MIN_DESIGN_PRECISION,'min_positive_folds':MIN_POSITIVE_DESIGN_FOLDS,'min_gain':MIN_DESIGN_GAIN,'bootstrap_ci_low_gt':MIN_DESIGN_BOOTSTRAP_LOWER},
    'lock_gates':{'min_precision':MIN_LOCK_PRECISION,'min_gain':MIN_LOCK_GAIN,'min_bootstrap_positive':MIN_LOCK_BOOTSTRAP_POSITIVE},
    'status':{'design_ok':DESIGN_OK,'lock_opened':LOCK_OPENED,'lock_ok':LOCK_OK,'validated':V8A_VALIDATED,'promoted':PROMOTED},
    'test':{'candidates':int(test_candidate.sum()),'rescues':int(test_rescue.sum()),'v5_negatives':int((test_v5_label==0).sum()),'v8a_proposed_negatives':int((proposed_v8a==0).sum())},
}
provenance = {
    'train_path':str(train_path),'test_path':str(test_path),'sample_path':str(sample_path),
    'train_id_sha256':ids_sha256(train.id),'test_id_sha256':ids_sha256(test.id),
    'frozen_artifact_sha256':_FROZEN_SHA256,'frozen_v5_submission_sha256':v5_sha,
    'timestamp_utc':pd.Timestamp.utcnow().isoformat(),
}
(OUTPUT_DIR/'config_v8a.json').write_text(json.dumps(config,indent=2,default=str))
(OUTPUT_DIR/'provenance_v8a.json').write_text(json.dumps(provenance,indent=2,default=str))

manifest={}
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file() and p.name not in {'manifest_v8a.json','results_v8a.zip'}:
        manifest[p.name]={'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()}
(OUTPUT_DIR/'manifest_v8a.json').write_text(json.dumps(manifest,indent=2))
with zipfile.ZipFile(OUTPUT_DIR/'results_v8a.zip','w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUTPUT_DIR.iterdir()):
        if p.is_file() and p.name!='results_v8a.zip': z.write(p,arcname=p.name)

print('Results bundle:', OUTPUT_DIR/'results_v8a.zip')
print('Final status:', config['status'])


Candidate selection rate DESIGN=16.7273% | test=16.3636%
Test candidates: 110 | proposed rescues: 18
Promoted: True
SUBMIT artifact created: /kaggle/working/ifest_v8a_output/submission_v8a.csv
Results bundle: /kaggle/working/ifest_v8a_output/results_v8a.zip
Final status: {'design_ok': True, 'lock_opened': True, 'lock_ok': True, 'validated': True, 'promoted': True}
